# Experiment 1 — Manifold Emergence Profile

**Goal.** For each model, measure the intrinsic dimension (ID) of the astronomical
embedding cloud at every layer, with scale-stability checks and null models.
Decide whether, and at which layer, a genuine low-dimensional manifold emerges.

**Outputs:** $L^*$ (the manifold layer) and $d^*$ (the plateau dimension) for
every model.

**Interpretation:** If $d^*$ lands near the number of physical parameters that
govern galaxies (roughly 5 to 10), far below both the raw-data dimension and the
matched Gaussian null, and is stable across scales, the model has compressed the
sky to physics-sized dimensionality. If the data ID is indistinguishable from the
Gaussian null at every layer, there is no manifold claim to make.

**Models (prototype):** 4 architectures at 021M scale:
`ar_affine_021M`, `ar_aim_021M`, `mae_affine_021M`, `mae_aim_021M`
— each with a trained and random-init (untrained) variant.
Swap `MODEL_LIST` to port to Yash's extracted embeddings later.

In [ ]:
# --- 0. Config & imports ---------------------------------------------------
from __future__ import annotations

import warnings
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy import linalg
from tqdm.notebook import tqdm

# ID estimation
try:
    from dadapy import Data
    HAS_DADAPY = True
except ImportError:
    print("[warn] DADApy not installed — GRIDE and DADApy TwoNN will not work.\n"
          "        Try: pip install dadapy")
    HAS_DADAPY = False

import skdim
from skdim.id import lPCA, TwoNN
from sklearn.random_projection import GaussianRandomProjection

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

print(f"Python {sys.version}")
print(f"numpy {np.__version__}")
print(f"skdim {skdim.__version__}")
print(f"DADApy available: {HAS_DADAPY}")

# ---------------------------------------------------------------------------
# SWAP THIS LIST when Yash's extracted embeddings arrive
# e.g. MODEL_LIST = ["pu_vit_021M", "pu_resnet_021M", ...]
# ---------------------------------------------------------------------------
MODEL_LIST = [
    "ar_affine_021M",
    "ar_aim_021M",
    "mae_affine_021M",
    "mae_aim_021M",
]
TAGS = ["trained", "untrained"]
EMB_DIR = Path("../reindex_embeddings")  # relative to metrics/

BOOT_N = 10       # bootstrap repeats
BOOT_SIZE = 400   # subsample size per bootstrap draw (N_total=500)
RNG_SEED = 42

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"
print(f"Configured {len(MODEL_LIST)} models, {TAGS}")

## 1. Load Embeddings

Load all parquet files, enforce identical row ordering, and separate the
`pixel_pca` column as a Layer-0 anchor.

In [ ]:
# --- 1. Load all parquets --------------------------------------------------

def load_one(path: Path):
    """Return (object_ids, {layer: (N,D) array})."""
    df = pl.read_parquet(path)
    oids = df["object_id"].to_numpy()
    layers = {c: np.array(df[c].to_list(), dtype=np.float64)
              for c in df.columns if c != "object_id"}
    return oids, layers

def ordered_layers(layers: dict) -> list[str]:
    """Sort: encoder, h.00..h.NN, ln_f, pixel_pca."""
    def keyfn(n):
        if n == "encoder":    return (-1, 0)
        if n == "ln_f":       return (10_000, 0)
        if n == "pixel_pca":  return (10_001, 0)
        if n.startswith("h."):
            return (int(n.split(".")[1]), 0)
        return (9_999, 0)
    return sorted(layers, key=keyfn)

ALL_DATA: dict[tuple[str, str], dict[str, np.ndarray]] = {}
ALL_ORDER: dict[tuple[str, str], list[str]] = {}
REF_IDS = None

for model in MODEL_LIST:
    for tag in TAGS:
        path = EMB_DIR / f"{model}_{tag}_blocks_layerwise.parquet"
        if not path.exists():
            print(f"[skip] {path.name} not found")
            continue
        oids, layers = load_one(path)
        if REF_IDS is None:
            REF_IDS = oids
        else:
            assert np.array_equal(oids, REF_IDS), \
                f"{model} [{tag}]: object_id mismatch!"
        ALL_DATA[(model, tag)] = layers
        ALL_ORDER[(model, tag)] = ordered_layers(layers)

N = len(REF_IDS) if REF_IDS is not None else 0
print(f"Loaded {len(ALL_DATA)} (model, tag) combos, N={N} galaxies each.")
for (m, t), order in ALL_ORDER.items():
    print(f"  {m:20s} [{t:9s}] -> {len(order)} layers: {order}")

## 2. Step 1 — TwoNN Profile with Error Bars

For each layer, compute TwoNN ID via DADApy (`Data(X).compute_id_2NN()`) and
participation ratio via `skdim.id.lPCA(ver='participation_ratio')`.
Repeat on 10 bootstrap subsamples of 400 rows (seeds 0–9) for error bars.

**Sanity check**: PR ≥ TwoNN at every layer.

In [ ]:
# --- 2. Step 1: TwoNN + PR profile -----------------------------------------

def twonn_profile(layers: dict, order: list[str],
                  bootstrap: bool = True) -> dict:
    """Return {layer_name: {twonn, twonn_err, pr, ...}}."""
    result = {}
    core = [n for n in order if n != "pixel_pca"]

    for name in tqdm(core, desc="TwoNN profile"):
        X = layers[name]
        # DADApy TwoNN
        if HAS_DADAPY:
            d_obj = Data(X, verbose=False)
            id_est, id_err, scale = d_obj.compute_id_2NN()
        else:
            id_est = TwoNN().fit(X).dimension_
            id_err = 0.0

        # Bootstrap error
        boot_vals = []
        if bootstrap and HAS_DADAPY:
            rng = np.random.default_rng(RNG_SEED)
            for s in range(BOOT_N):
                idx = rng.choice(N, size=min(BOOT_SIZE, N), replace=True)
                Xb = X[idx]
                db = Data(Xb, verbose=False)
                idb, _, _ = db.compute_id_2NN()
                boot_vals.append(idb)
        elif bootstrap:
            rng = np.random.default_rng(RNG_SEED)
            for s in range(BOOT_N):
                idx = rng.choice(N, size=min(BOOT_SIZE, N), replace=True)
                idb = TwoNN().fit(X[idx]).dimension_
                boot_vals.append(idb)

        # Participation ratio (linear dimension)
        pr = lPCA(ver='participation_ratio').fit(X).dimension_

        rec = {
            "twonn": id_est,
            "twonn_err": id_err,
            "boot_mean": float(np.mean(boot_vals)) if boot_vals else id_est,
            "boot_std": float(np.std(boot_vals)) if boot_vals else id_err,
            "pr": pr,
        }
        result[name] = rec

    # Sanity check
    for name, rec in result.items():
        if rec["pr"] < rec["twonn"] - 1e-6:
            print(f"[WARN] {name}: PR={rec['pr']:.1f} < TwoNN={rec['twonn']:.1f} — "
                  f"possible float16 underflow or duplicate rows")
    return result


def plot_profile(profiles: dict, model: str, tag: str,
                 ax: plt.Axes = None) -> plt.Axes:
    """Plot TwoNN + PR vs layer index."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    core = [n for n in profiles if n != "pixel_pca"]
    idx = np.arange(len(core))
    tw = np.array([profiles[n]["twonn"] for n in core])
    te = np.array([profiles[n]["boot_std"] for n in core])
    pr = np.array([profiles[n]["pr"] for n in core])

    ax.errorbar(idx, tw, yerr=te, fmt='o-', color='tab:blue',
                capsize=3, label='TwoNN', markersize=5)
    ax.plot(idx, pr, 's--', color='tab:orange', label='Participation ratio',
            markersize=4)
    ax.set_xticks(idx)
    ax.set_xticklabels(core, rotation=45, fontsize=8)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Intrinsic dimension")
    ax.set_title(f"{model} [{tag}]")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    return ax


# --- Compute for all models ------------------------------------------------
PROFILES: dict[tuple[str, str], dict] = {}
for (model, tag) in tqdm(ALL_DATA, desc="Profiles"):
    PROFILES[(model, tag)] = twonn_profile(
        ALL_DATA[(model, tag)], ALL_ORDER[(model, tag)]
    )

# --- Plot ------------------------------------------------------------------
n_models = len(MODEL_LIST)
fig, axes = plt.subplots(2, n_models, figsize=(4.5 * n_models, 8), squeeze=False)
for i, model in enumerate(MODEL_LIST):
    for j, tag in enumerate(TAGS):
        key = (model, tag)
        if key in PROFILES:
            plot_profile(PROFILES[key], model, tag, ax=axes[j][i])
        else:
            axes[j][i].text(0.5, 0.5, "N/A", ha='center', va='center')
    if i == 0:
        axes[0][i].set_ylabel("Trained")
        axes[1][i].set_ylabel("Untrained")
plt.tight_layout()
plt.show()

## 3. Step 2 — Scale Curve with GRIDE

At every layer, compute the ID-versus-scale curve with DADApy's GRIDE:
`ids, errs, scales = d.return_id_scaling_gride(range_max=128)`.

**Plateau rule:** find the longest run of at least 3 consecutive scale points
where `max(ids) - min(ids) < 0.10 * mean(ids)`. If such a run exists, the
plateau ID is the mean over the run. If no run exists at any layer, the model
gets no manifold layer.

In [ ]:
# --- 3. Step 2: GRIDE scale curves -----------------------------------------

def gride_plateau(layers: dict, order: list[str]) -> dict:
    """Run GRIDE per layer; detect plateaus.
    Returns {layer_name: {ids, errs, scales, plateau_exists, plateau_id, plateau_mask}}
    or None for layers where GRIDE fails.
    """
    core = [n for n in order if n != "pixel_pca"]
    results = {}
    for name in tqdm(core, desc="GRIDE"):
        X = layers[name]
        d_obj = Data(X, verbose=False)
        try:
            ids, errs, scales = d_obj.return_id_scaling_gride(range_max=128)
        except Exception as e:
            print(f"  {name}: GRIDE failed — {e}")
            results[name] = None
            continue

        ids = np.asarray(ids)
        errs = np.asarray(errs)
        scales = np.asarray(scales)

        # Detect plateau
        plateau_exists = False
        plateau_id = np.nan
        plateau_mask = np.zeros(len(ids), dtype=bool)

        if len(ids) >= 3:
            best_len = 0
            best_start = 0
            for start in range(len(ids) - 2):
                for end in range(start + 2, len(ids)):
                    segment = ids[start:end + 1]
                    spread = segment.max() - segment.min()
                    if spread < 0.10 * segment.mean():
                        run_len = end - start + 1
                        if run_len > best_len:
                            best_len = run_len
                            best_start = start
            if best_len >= 3:
                plateau_exists = True
                plateau_mask[best_start:best_start + best_len] = True
                plateau_id = float(ids[plateau_mask].mean())

        results[name] = {
            "ids": ids,
            "errs": errs,
            "scales": scales,
            "plateau_exists": plateau_exists,
            "plateau_id": plateau_id,
            "plateau_mask": plateau_mask,
        }
    return results


def plot_gride(results: dict, model: str, tag: str,
               ax: plt.Axes = None) -> plt.Axes:
    """Plot GRIDE curves for layers with plateaus."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    for name, r in results.items():
        if r is None:
            continue
        label = name
        marker = 'o' if r['plateau_exists'] else 'x'
        color = 'tab:green' if r['plateau_exists'] else 'tab:gray'
        ax.errorbar(r['scales'], r['ids'], yerr=r['errs'],
                    fmt=marker + '-', color=color, alpha=0.7,
                    label=label, markersize=4, capsize=2)
        if r['plateau_exists']:
            mask = r['plateau_mask']
            ax.axhline(r['plateau_id'], color=color, ls=':', alpha=0.5,
                       xmin=r['scales'][mask].min() / r['scales'].max(),
                       xmax=r['scales'][mask].max() / r['scales'].max())
    ax.set_xlabel("Scale (neighbor range)")
    ax.set_ylabel("ID")
    ax.set_title(f"GRIDE scale curves — {model} [{tag}]")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=0.3)
    return ax


# --- Compute GRIDE for all models ------------------------------------------
GRIDE_RESULTS: dict[tuple[str, str], dict] = {}
for (model, tag) in tqdm(ALL_DATA, desc="GRIDE all models"):
    GRIDE_RESULTS[(model, tag)] = gride_plateau(
        ALL_DATA[(model, tag)], ALL_ORDER[(model, tag)]
    )

# --- Print plateau summary -------------------------------------------------
print("\n=== Plateau summary ===")
for (model, tag), results in GRIDE_RESULTS.items():
    n_plateau = sum(1 for r in results.values()
                    if r is not None and r['plateau_exists'])
    print(f"{model:20s} [{tag:9s}]: {n_plateau} layers with plateaus")
    for name, r in results.items():
        if r is not None and r['plateau_exists']:
            print(f"    {name:10s} -> plateau ID = {r['plateau_id']:.2f}")

## 4. Step 3 — Decimation Curve

At each candidate layer (layers with a plateau), run DADApy's subsampling
curve: `ids_dec, errs_dec, scales_dec = d.return_id_scaling_2NN(N_min=500)`.
Fit a straight line (numpy polyfit) to ID vs log(subset size).

Require the absolute slope to be less than `0.05 * plateau_ID`.
A drifting decimation curve overrules the plateau.

In [ ]:
# --- 4. Step 3: Decimation curves ------------------------------------------

def decimation_check(layers: dict, order: list[str],
                     gride_results: dict) -> dict:
    """Run decimation on candidate layers (those with plateaus).
    Returns {layer_name: {dec_ids, dec_scales, slope, passes, plateau_id}}.
    """
    core = [n for n in order if n != "pixel_pca"]
    results = {}
    for name in tqdm(core, desc="Decimation"):
        gr = gride_results.get(name)
        if gr is None or not gr['plateau_exists']:
            continue
        plateau_id = gr['plateau_id']

        X = layers[name]
        d_obj = Data(X, verbose=False)
        try:
            ids, errs, scales = d_obj.return_id_scaling_2NN(N_min=500)
        except Exception as e:
            print(f"  {name}: decimation failed — {e}")
            continue

        ids = np.asarray(ids)
        scales = np.asarray(scales)
        log_n = np.log(np.maximum(scales, 1))
        slope, intercept = np.polyfit(log_n, ids, 1)
        passes = abs(slope) < 0.05 * plateau_id

        results[name] = {
            "dec_ids": ids,
            "dec_errs": errs,
            "dec_scales": scales,
            "slope": slope,
            "passes": passes,
            "plateau_id": plateau_id,
        }
    return results


def plot_decimation(dec_results: dict, model: str, tag: str,
                    ax: plt.Axes = None) -> plt.Axes:
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    for name, r in dec_results.items():
        color = 'tab:green' if r['passes'] else 'tab:red'
        marker = 'o' if r['passes'] else 'x'
        ax.plot(r['dec_scales'], r['dec_ids'], f'{marker}-',
                color=color, label=f"{name} (slope={r['slope']:.3f})")
    ax.axhline(y=0, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel("Subset size")
    ax.set_ylabel("ID")
    ax.set_title(f"Decimation — {model} [{tag}]")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    return ax


# --- Compute decimation for all models -------------------------------------
DEC_RESULTS: dict[tuple[str, str], dict] = {}
for (model, tag) in tqdm(ALL_DATA, desc="Decimation all models"):
    DEC_RESULTS[(model, tag)] = decimation_check(
        ALL_DATA[(model, tag)], ALL_ORDER[(model, tag)],
        GRIDE_RESULTS[(model, tag)]
    )

# --- Print decimation summary ----------------------------------------------
print("\n=== Decimation summary ===")
for (model, tag), results in DEC_RESULTS.items():
    n_pass = sum(1 for r in results.values() if r['passes'])
    n_total = len(results)
    print(f"{model:20s} [{tag:9s}]: {n_pass}/{n_total} candidate layers pass decimation")
    for name, r in results.items():
        status = "PASS" if r['passes'] else "FAIL"
        print(f"    {name:10s} slope={r['slope']:.4f}  threshold={0.05*r['plateau_id']:.4f}  [{status}]")

## 5. Step 4 — Nulls

For every layer, build two fakes with the same rows:

1. **Gaussian-spectrum null**: fit the empirical mean and covariance and draw
   N samples with `numpy.random.default_rng(0).multivariate_normal(mean, cov)`.
2. **Column-shuffle null**: permute each column of X independently.

Re-run Steps 1 and 2 (TwoNN + GRIDE) on both fakes.

**Manifold criterion** at a layer: all of:
- (a) a data plateau exists
- (b) data plateau ID < Gaussian-null ID at same layer by >3 combined standard errors
- (c) decimation slope passes Step 3

In [ ]:
# --- 5. Step 4: Nulls ------------------------------------------------------

def build_nulls(X: np.ndarray, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """Return (gaussian_null, shuffle_null) with same shape as X."""
    rng = np.random.default_rng(seed)
    # Gaussian-spectrum null
    mean = X.mean(axis=0)
    cov = np.cov(X, rowvar=False)
    # Regularize cov for numerical stability
    cov += 1e-8 * np.eye(cov.shape[0])
    try:
        gauss = rng.multivariate_normal(mean, cov, method='eigh')
    except Exception as e:
        print(f"    multivariate_normal failed: {e}, falling back to cholesky")
        gauss = rng.multivariate_normal(mean, cov, method='cholesky')

    # Column-shuffle null
    shuffle = X.copy()
    for col in range(shuffle.shape[1]):
        rng.shuffle(shuffle[:, col])
    return gauss, shuffle


def null_analysis(layers: dict, order: list[str],
                  gride_results: dict,
                  dec_results: dict) -> dict:
    """For each layer, compute Gaussian-null and shuffle-null GRIDE plateaus.
    Returns {layer_name: {gauss_plateau_id, gauss_plateau_err,
                          shuffle_plateau_id, passes_criterion}}.
    """
    core = [n for n in order if n != "pixel_pca"]
    results = {}
    for name in tqdm(core, desc="Nulls"):
        X = layers[name]
        gr = gride_results.get(name)
        dr = dec_results.get(name)

        # Check criterion (a) and (c) first
        data_plateau = gr is not None and gr['plateau_exists']
        dec_passes = dr is not None and dr['passes']

        gauss, shuffle = build_nulls(X, seed=0)

        # GRIDE on Gaussian null
        gauss_plateau_id = np.nan
        gauss_plateau_err = np.nan
        try:
            dg = Data(gauss, verbose=False)
            g_ids, g_errs, g_scales = dg.return_id_scaling_gride(range_max=128)
            # Detect plateau on null
            g_ids = np.asarray(g_ids)
            if len(g_ids) >= 3:
                best_len = 0
                best_start = 0
                for start in range(len(g_ids) - 2):
                    for end in range(start + 2, len(g_ids)):
                        seg = g_ids[start:end + 1]
                        if seg.max() - seg.min() < 0.10 * seg.mean():
                            rl = end - start + 1
                            if rl > best_len:
                                best_len = rl
                                best_start = start
                if best_len >= 3:
                    gauss_plateau_id = float(g_ids[best_start:best_start + best_len].mean())
                    gauss_plateau_err = float(np.mean(g_errs[best_start:best_start + best_len]))
        except Exception as e:
            print(f"  {name}: Gaussian null GRIDE failed — {e}")

        # GRIDE on shuffle null
        shuffle_plateau_id = np.nan
        try:
            ds = Data(shuffle, verbose=False)
            s_ids, s_errs, s_scales = ds.return_id_scaling_gride(range_max=128)
            s_ids = np.asarray(s_ids)
            if len(s_ids) >= 3:
                best_len = 0
                best_start = 0
                for start in range(len(s_ids) - 2):
                    for end in range(start + 2, len(s_ids)):
                        seg = s_ids[start:end + 1]
                        if seg.max() - seg.min() < 0.10 * seg.mean():
                            rl = end - start + 1
                            if rl > best_len:
                                best_len = rl
                                best_start = start
                if best_len >= 3:
                    shuffle_plateau_id = float(s_ids[best_start:best_start + best_len].mean())
        except Exception as e:
            print(f"  {name}: shuffle null GRIDE failed — {e}")

        # Criterion (b): data plateau ID < Gaussian-null ID by >3 combined SE
        passes_criterion = False
        if data_plateau and not np.isnan(gauss_plateau_id) and not np.isnan(gr['plateau_id']):
            data_id = gr['plateau_id']
            data_err = float(np.mean(gr['errs'][gr['plateau_mask']])) if gr['plateau_mask'].any() else 0.0
            diff = gauss_plateau_id - data_id
            combined_err = np.sqrt(data_err**2 + gauss_plateau_err**2)
            if combined_err > 0 and diff / combined_err > 3:
                passes_criterion = True

        results[name] = {
            "data_plateau": data_plateau,
            "data_plateau_id": gr['plateau_id'] if data_plateau else np.nan,
            "gauss_plateau_id": gauss_plateau_id,
            "gauss_plateau_err": gauss_plateau_err,
            "shuffle_plateau_id": shuffle_plateau_id,
            "dec_passes": dec_passes,
            "passes_criterion": passes_criterion,
        }
    return results


# --- Compute null analysis for all models ----------------------------------
NULL_RESULTS: dict[tuple[str, str], dict] = {}
for (model, tag) in tqdm(ALL_DATA, desc="Null analysis"):
    NULL_RESULTS[(model, tag)] = null_analysis(
        ALL_DATA[(model, tag)], ALL_ORDER[(model, tag)],
        GRIDE_RESULTS[(model, tag)], DEC_RESULTS[(model, tag)]
    )

# --- Print null summary ----------------------------------------------------
print("\n=== Null criterion summary ===")
for (model, tag), results in NULL_RESULTS.items():
    n_pass = sum(1 for r in results.values() if r['passes_criterion'])
    print(f"{model:20s} [{tag:9s}]: {n_pass} layers pass full manifold criterion")
    for name, r in results.items():
        if r['passes_criterion']:
            print(f"    {name:10s}: data_id={r['data_plateau_id']:.2f}  "
                  f"gauss_null_id={r['gauss_plateau_id']:.2f}  "
                  f"shuffle_null_id={r['shuffle_plateau_id']:.2f}")

## 6. Step 5 — Anchors

Compute two reference IDs and draw them as horizontal lines on the profile figure.

1. **Raw data anchor**: flatten pixels (from pixel_pca or raw images), random-project
   to 4096 dimensions with `sklearn.random_projection.GaussianRandomProjection`,
   and run DADApy TwoNN.
2. **Native-domain profile**: placeholder — requires pretraining-domain data
   (ImageNet-like images) which is not available in this prototype.

In [ ]:
# --- 6. Step 5: Anchors ----------------------------------------------------

def raw_data_anchor(layers: dict, order: list[str]) -> float:
    """Compute TwoNN ID on random-projected pixel data.
    Uses pixel_pca if available, otherwise falls back to first layer.
    """
    if "pixel_pca" in layers:
        X_pix = layers["pixel_pca"]
    else:
        # Use first available layer as proxy
        first = [n for n in order if n != "pixel_pca"][0]
        X_pix = layers[first]
        print(f"[warn] pixel_pca not found, using {first} as raw anchor proxy")

    # Random projection to 4096D
    n_components = min(4096, X_pix.shape[1], X_pix.shape[0] - 1)
    rp = GaussianRandomProjection(n_components=n_components, random_state=0)
    X_rp = rp.fit_transform(X_pix)

    if HAS_DADAPY:
        d = Data(X_rp, verbose=False)
        id_est, _, _ = d.compute_id_2NN()
    else:
        id_est = TwoNN().fit(X_rp).dimension_
    print(f"Raw data anchor (random-projected to {n_components}D): TwoNN = {id_est:.2f}")
    return id_est


# Compute raw data anchor (same for all models since pixel_pca is shared)
first_key = next(iter(ALL_DATA))
RAW_ANCHOR = raw_data_anchor(ALL_DATA[first_key], ALL_ORDER[first_key])

# --- Re-plot profiles with anchor lines ------------------------------------
fig, axes = plt.subplots(2, n_models, figsize=(4.5 * n_models, 8), squeeze=False)
for i, model in enumerate(MODEL_LIST):
    for j, tag in enumerate(TAGS):
        key = (model, tag)
        if key in PROFILES:
            ax = plot_profile(PROFILES[key], model, tag, ax=axes[j][i])
            ax.axhline(RAW_ANCHOR, color='red', ls='--', alpha=0.7,
                       label=f'Raw anchor ({RAW_ANCHOR:.1f})')
            ax.legend(fontsize=7)
        else:
            axes[j][i].text(0.5, 0.5, "N/A", ha='center', va='center')
    if i == 0:
        axes[0][i].set_ylabel("Trained")
        axes[1][i].set_ylabel("Untrained")
plt.tight_layout()
plt.show()

## 7. Step 6 — Define $L^*$ and $d^*$

For each model:
- $L^*$ is the earliest layer that passes the full manifold criterion of Step 4
  and whose plateau ID is within 20 percent of the minimum plateau ID over all
  passing layers.
- $d^*$ is the plateau ID at $L^*$.

Produce a table with columns: model, $L^*$, $L^*/(L-1)$, $d^*$, Gaussian-null ID
at $L^*$, and participation ratio at $L^*$.

In [ ]:
# --- 7. Step 6: L* and d* table --------------------------------------------

def compute_L_star(model: str, tag: str) -> dict | None:
    """Compute L* and d* for one (model, tag). Returns dict or None."""
    null_r = NULL_RESULTS.get((model, tag), {})
    gride_r = GRIDE_RESULTS.get((model, tag), {})
    prof = PROFILES.get((model, tag), {})
    order = ALL_ORDER.get((model, tag), [])
    core = [n for n in order if n != "pixel_pca"]

    # Find all passing layers
    passing = []
    for name in core:
        nr = null_r.get(name)
        if nr is not None and nr['passes_criterion']:
            passing.append((name, nr['data_plateau_id']))

    if not passing:
        return None

    # Minimum plateau ID among passing layers
    min_id = min(pid for _, pid in passing)

    # Earliest layer within 20% of min
    for name, pid in passing:
        if pid <= 1.20 * min_id:
            L_star = name
            d_star = pid
            break
    else:
        L_star = passing[0][0]
        d_star = passing[0][1]

    # Layer index
    L_idx = core.index(L_star)
    L_total = len(core)

    # Gaussian null ID at L*
    gauss_id = null_r[L_star]['gauss_plateau_id']

    # Participation ratio at L*
    pr_val = prof.get(L_star, {}).get('pr', np.nan)

    return {
        "model": model,
        "tag": tag,
        "L*": L_star,
        "L*_index": L_idx,
        "L*/(L-1)": L_idx / max(L_total - 1, 1),
        "d*": d_star,
        "Gaussian-null ID at L*": gauss_id,
        "PR at L*": pr_val,
        "n_passing_layers": len(passing),
    }


# --- Build table -----------------------------------------------------------
rows = []
for model in MODEL_LIST:
    for tag in TAGS:
        rec = compute_L_star(model, tag)
        if rec is not None:
            rows.append(rec)
        else:
            rows.append({
                "model": model, "tag": tag,
                "L*": "NONE", "L*_index": -1,
                "L*/(L-1)": np.nan, "d*": np.nan,
                "Gaussian-null ID at L*": np.nan,
                "PR at L*": np.nan,
                "n_passing_layers": 0,
            })

import pandas as pd
L_star_table = pd.DataFrame(rows)
print("=== L* and d* Table ===")
display(L_star_table)

## 8. Step 7 — Controls

Run the identical pipeline on the untrained control of every architecture.
Expected outcome: no layer passes the manifold criterion, or plateaus sit near
the null. If an untrained model produces a clean $L^*$, report it prominently.

In [ ]:
# --- 8. Step 7: Controls ---------------------------------------------------

print("=== Untrained Control Summary ===")
untrained_rows = L_star_table[L_star_table['tag'] == 'untrained']
n_with_L = (untrained_rows['n_passing_layers'] > 0).sum()
print(f"Untrained models with a valid L*: {n_with_L}/{len(untrained_rows)}")

if n_with_L > 0:
    print("\n[WARNING] Untrained models produce a manifold layer!")
    print("This means the criterion is detecting architecture + input statistics,")
    print("not learning. Thresholds must be treated as suspect for trained models.")
    display(untrained_rows[untrained_rows['n_passing_layers'] > 0])
else:
    print("✓ No untrained model produces a clean L* — criterion is specific to learning.")

# --- Trained vs untrained comparison ---------------------------------------
print("\n=== Trained vs Untrained Comparison ===")
for model in MODEL_LIST:
    tr = L_star_table[(L_star_table['model'] == model) & (L_star_table['tag'] == 'trained')]
    un = L_star_table[(L_star_table['model'] == model) & (L_star_table['tag'] == 'untrained')]
    tr_d = tr['d*'].values[0] if len(tr) > 0 else np.nan
    un_d = un['d*'].values[0] if len(un) > 0 else np.nan
    print(f"  {model:20s}: trained d*={tr_d:.2f}  untrained d*={un_d:.2f}")

## 9. Step 8 — Summary

Concise summary stating whether a manifold emerges, at which relative depth,
at what dimension, how $d^*$ compares to the count of physical parameters
(roughly 5–10), and whether the result survives the untrained control.

In [ ]:
# --- 9. Step 8: Summary ----------------------------------------------------

trained_rows = L_star_table[L_star_table['tag'] == 'trained']
n_models_with_L = (trained_rows['n_passing_layers'] > 0).sum()
n_models_total = len(trained_rows)

print("=" * 70)
print("EXPERIMENT 1 — MANIFOLD EMERGENCE PROFILE: SUMMARY")
print("=" * 70)
print()

if n_models_with_L == 0:
    print("RESULT: No trained model produces a valid manifold layer.")
    print("The data ID is indistinguishable from the Gaussian null at every layer.")
    print("There is no manifold claim to make — Experiments 3 and 4 must be reported")
    print("as negative.")
else:
    print(f"RESULT: {n_models_with_L}/{n_models_total} trained models produce a valid manifold layer.")
    print()
    for _, row in trained_rows.iterrows():
        if row['n_passing_layers'] > 0:
            print(f"  {row['model']:20s}:")
            print(f"    L* = {row['L*']:10s}  (relative depth = {row['L*/(L-1)']:.2f})")
            print(f"    d* = {row['d*']:.2f}")
            print(f"    Gaussian-null ID at L* = {row['Gaussian-null ID at L*']:.2f}")
            print(f"    Participation ratio at L* = {row['PR at L*']:.2f}")
            print()

    # Compare d* to physical parameters (5-10)
    d_stars = trained_rows.loc[trained_rows['n_passing_layers'] > 0, 'd*']
    if len(d_stars) > 0:
        mean_d = d_stars.mean()
        print(f"  Mean d* across models: {mean_d:.1f}")
        if 5 <= mean_d <= 10:
            print(f"  d* = {mean_d:.1f} lands near the 5-10 physical parameter range.")
            print("  The model has compressed the sky to physics-sized dimensionality.")
        elif mean_d < 5:
            print(f"  d* = {mean_d:.1f} is below the 5-10 physical parameter range.")
            print("  The representation is even more compressed than expected.")
        else:
            print(f"  d* = {mean_d:.1f} is above the 5-10 physical parameter range.")
            print("  The representation is not fully compressed to physics scale.")

print()
print("--- Control check ---")
untrained_with_L = (L_star_table[L_star_table['tag'] == 'untrained']['n_passing_layers'] > 0).sum()
if untrained_with_L > 0:
    print(f"WARNING: {untrained_with_L} untrained models produce a valid L*.")
    print("The manifold criterion may be detecting architecture + input statistics,")
    print("not learning. Thresholds should be re-examined.")
else:
    print("✓ No untrained model produces a valid L* — result survives control.")

print()
print("=" * 70)

## Next Steps

1. **Swap MODEL_LIST** at the top when Yash's extracted embeddings arrive.
2. **Native-domain anchor**: requires pretraining-domain data (ImageNet-like).
   Add when available.
3. **Cross-check with skdim TwoNN** at 3 representative layers if DADApy results
   look suspicious.
4. **Scale up** to all 12 models (or however many Yash extracts) by expanding
   MODEL_LIST.